## Exploratory longitudinal association

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import silhouette_score, roc_auc_score, confusion_matrix
from scipy import stats

In [ ]:
# Runs the clustering pipeline first (set GROUP = "CM" in that notebook)
%run migraine_clustering_pipeline.ipynb

In [ ]:
assert GROUP == "CM", "Set GROUP = 'CM' to run the longitudinal analysis"

In [ ]:
def make_logreg():
    return LogisticRegression(max_iter=1000, random_state=SEED)


def loo_auc_and_cm(X, y):
    loo = LeaveOneOut()
    y_proba = np.zeros(len(y))
    y_pred = np.zeros(len(y))
    for tr, te in loo.split(X):
        m = make_logreg().fit(X[tr], y[tr])
        y_proba[te] = m.predict_proba(X[te])[:, 1]
        y_pred[te] = m.predict(X[te])
    return roc_auc_score(y, y_proba), confusion_matrix(y, y_pred)


def likelihood_ratio_test(X_full, X_base, y):
    def loglik(X, y):
        m = make_logreg().fit(X, y)
        p = np.clip(m.predict_proba(X), 1e-10, 1 - 1e-10)
        return np.sum(y * np.log(p[:, 1]) + (1 - y) * np.log(p[:, 0]))
    lr = 2 * (loglik(X_full, y) - loglik(X_base, y))
    return stats.chi2.sf(lr, df=1)


def longitudinal_association(clin_cm, cluster_labels, baseline_vars, outcome_col):
    d = clin_cm.join(cluster_labels, how="inner")      
    assert len(d) == len(clin_cm) 
    d = d[d[outcome_col].notna()]
    y = d[outcome_col].astype(int).values
    X_base = d[baseline_vars].values
    X_full = np.hstack([X_base, d[["cluster"]].values])

    auc_base, cm_base = loo_auc_and_cm(X_base, y)
    auc_full, cm_full = loo_auc_and_cm(X_full, y)
    p = likelihood_ratio_test(X_full, X_base, y)
    return {"n": len(y), "auc_base": auc_base, "auc_full": auc_full,
            "delta_auc": auc_full - auc_base, "p_lrt": p,
            "cm_base": cm_base, "cm_full": cm_full}

In [ ]:
clin_cm = clin_df[clin_df["CM_base"] == 1]

# Baseline clinical variables included in all models
baseline = ["Woman", "Age", "MOH", "Aura", "freq_mig", "time_mig", "time_cro"]

# Cluster labels used as predictor 
cluster_results = results

for outcome in ("CM_long", "topiramate_binary"):
    print(f"\nOUTCOME: {outcome}")
    for method, r in cluster_results.items():
        res = longitudinal_association(clin_cm, r["labels"], baseline, outcome)
        print(f"  {method}: N={res['n']} | AUC base={res['auc_base']:.3f} | "
              f"AUC with cluster={res['auc_full']:.3f} | "
              f"dAUC={res['delta_auc']:+.3f} | p(LRT)={res['p_lrt']:.3f}")
        print(f"    Confusion matrix (with cluster): {res['cm_full'].tolist()}")